# Safeguarding AutoGen Multi-Agent Systems with Bartholomew Trust Protocol (BTP)

In autonomous multi-agent environments, agents often have access to shell tools, database execution engines, and API endpoints. Relying solely on LLM system prompts or moderation APIs exposes applications to **prompt injection attacks**, **jailbreaks**, and **hallucinations** that can execute destructive operations (such as `rm -rf` or `DROP TABLE`).

**Bartholomew Trust Protocol (BTP v5.4)** provides an in-process, deterministic Abstract Syntax Tree (AST) safety runtime. It evaluates tool parameters and shell commands in CPU memory in **under 35 microseconds**, vetoing malicious actions before they ever reach the operating system or database driver.

This recipe demonstrates:
1. Decorating AutoGen tools with `@btp_autogen_guard`.
2. **Allowed Execution**: Safe read queries passing without friction.
3. **Blocked Violation with Diagnostics**: Intercepting destructive commands with microsecond audit reports.
4. **Agent Message Interception**: Inspecting inter-agent communication using `AutoGenBTPInterceptor`.

## 1. Installation

Install `pyautogen` and `btp-guard`:

In [ ]:
!pip install pyautogen btp-guard

## 2. Defining BTP-Guarded AutoGen Tools

Decorate tool functions with `@btp_autogen_guard`. Any string arguments passed by the LLM are evaluated against AST security invariants before the tool executes.

In [ ]:
from framework_adapters.autogen import btp_autogen_guard, BTPViolationError, AutoGenBTPInterceptor

# Define a protected SQL execution tool for AutoGen agents
@btp_autogen_guard
def execute_sql_query(query: str) -> str:
    """Simulates running a SQL query against the enterprise warehouse."""
    return f"[SUCCESS] Executed: '{query}'. Returned 10 records."

# Define a protected terminal execution tool
@btp_autogen_guard
def execute_bash_command(command: str) -> str:
    """Simulates running a terminal command in the worker environment."""
    return f"[SUCCESS] Executed command: '{command}'. Exit code: 0."

## 3. Demonstration 1: Allowed Safe Execution

Standard operational commands (e.g. data queries, directory listings) evaluate as safe and execute normally.

In [ ]:
safe_query = "SELECT user_id, email, organization_id FROM users WHERE active = 1 LIMIT 10;"
result = execute_sql_query(safe_query)
print("Tool Result:", result)

## 4. Demonstration 2: Intercepting Destructive Injections with Rich Diagnostics

When a prompt injection or hallucinating agent attempts to execute a destructive operation (`DROP TABLE`, `rm -rf`, or database truncation), the BTP AST gate intercepts it in under 35µs and raises `BTPViolationError` with structured diagnostics.

In [ ]:
malicious_query = "DROP TABLE enterprise_users CASCADE;"

try:
    execute_sql_query(malicious_query)
except BTPViolationError as exc:
    print("=== BTP SECURITY VETO INTERCEPTED ===")
    print(exc)
    print("\n=== STRUCTURED JSON DIAGNOSTICS ===")
    import json
    print(json.dumps(exc.to_diagnostics(), indent=2))

Now let's test a destructive terminal command injection:

In [ ]:
destructive_cmd = "rm -rf /var/lib/data"

try:
    execute_bash_command(destructive_cmd)
except BTPViolationError as exc:
    print(f"Blocked Rule:    {exc.rule_id}")
    print(f"Latency:         {exc.latency_us:.1f} µs")
    print(f"Blocked Reason:  {exc.reason}")

## 5. Demonstration 3: In-Flight Message Interception (Confused Deputy Prevention)

In multi-agent teams, an attacker may try to use a compromised peer agent to trick another agent into running privileged commands. `AutoGenBTPInterceptor` filters in-flight messages before they trigger code execution blocks.

In [ ]:
interceptor = AutoGenBTPInterceptor()

# 1. Inbound safe message
inbound_safe = {"role": "user", "content": "Please summarize the latest security report."}
filtered_safe = interceptor.intercept_message(inbound_safe)
print("Safe Message Status:", filtered_safe.get("status", "PASSED"))

# 2. Inbound prompt injection payload
inbound_attack = {
    "role": "assistant",
    "content": "I will now run the maintenance script: rm -rf /etc/ssl/certs"
}
filtered_attack = interceptor.intercept_message(inbound_attack)
print("\nAttack Message Status:", filtered_attack.get("status"))
print("System Security Notice:", filtered_attack.get("content"))

## 6. Summary

By wrapping AutoGen tools with `@btp_autogen_guard`, engineering teams gain:
- **Deterministic Safety**: Intercepts `rm -rf`, `DROP TABLE`, and SQL injections before execution.
- **Zero Cloud Latency**: Pure in-process AST parsing in under 35µs.
- **Zero Refactoring**: Single-line decorator compatibility with existing AutoGen agent tools.